# Image Classification with a CNN and a DNN (CIFAR-10)

Companion code for the report *Image Classification with a CNN and a DNN*.
Trains a Dense Neural Network and a regularized Convolutional Neural Network (with BatchNormalization, Dropout, GlobalAveragePooling, and L2 regularization) on the CIFAR-10 dataset.


## Imports and GPU setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Check & enable GPU
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


## Load CIFAR-10 and normalize

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize data
x_train_normalized = x_train / 255.0
x_test_normalized = x_test / 255.0

x_train_flat = x_train_normalized.reshape(len(x_train_normalized), -1)
x_test_flat = x_test_normalized.reshape(len(x_test_normalized), -1)


## Build models (DNN and regularized CNN)

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)]

with tf.device('/GPU:0' if gpus else '/CPU:0'):
    # DNN
    dnn_model = models.Sequential([
        layers.Input(shape=(x_train_flat.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(10)
    ])
    dnn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # CNN (regularized)
    cnn_model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(10)
    ])
    cnn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )


## Train both models

In [ ]:
with tf.device('/GPU:0' if gpus else '/CPU:0'):
    print("\n=== Training DNN ===")
    dnn_history = dnn_model.fit(
        x_train_flat, y_train,
        epochs=100, batch_size=128,
        validation_data=(x_test_flat, y_test),
        callbacks=callbacks
    )

    print("\n=== Training CNN ===")
    cnn_history = cnn_model.fit(
        x_train_normalized, y_train,
        epochs=100, batch_size=128,
        validation_data=(x_test_normalized, y_test),
        callbacks=callbacks
    )


## Loss curves

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dnn_history.history['loss'], label='train')
axes[0].plot(dnn_history.history['val_loss'], label='val')
axes[0].set_title('DNN Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(cnn_history.history['loss'], label='train')
axes[1].plot(cnn_history.history['val_loss'], label='val')
axes[1].set_title('CNN Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()


## Evaluation, predictions, classification report

In [ ]:
dnn_loss, dnn_acc = dnn_model.evaluate(x_test_flat, y_test, verbose=0)
cnn_loss, cnn_acc = cnn_model.evaluate(x_test_normalized, y_test, verbose=0)
print(f"DNN -> loss: {dnn_loss:.4f} | accuracy: {dnn_acc:.4f}")
print(f"CNN -> loss: {cnn_loss:.4f} | accuracy: {cnn_acc:.4f}")

dnn_preds = np.argmax(dnn_model.predict(x_test_flat, verbose=0), axis=1)
cnn_preds = np.argmax(cnn_model.predict(x_test_normalized, verbose=0), axis=1)
y_true = np.array(y_test).flatten()

print("\n=== DNN report ===")
print(classification_report(y_true, dnn_preds, target_names=class_names))
print("=== CNN report ===")
print(classification_report(y_true, cnn_preds, target_names=class_names))


## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay(
    confusion_matrix(y_true, dnn_preds),
    display_labels=class_names
).plot(ax=axes[0], cmap='Blues', colorbar=False, xticks_rotation=45)
axes[0].set_title(f'DNN (acc={dnn_acc:.3f})')

ConfusionMatrixDisplay(
    confusion_matrix(y_true, cnn_preds),
    display_labels=class_names
).plot(ax=axes[1], cmap='Blues', colorbar=False, xticks_rotation=45)
axes[1].set_title(f'CNN (acc={cnn_acc:.3f})')

plt.tight_layout(); plt.show()
